In [1]:
import gymnasium as gym
import numpy as np

import torch
import torch.nn as nn


import random
import os


In [ ]:

class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, checkpoint_dir = "./sac", file_name = "policy.pth") -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 500),
            nn.ReLU(),
            nn.Linear(500, 100),
            nn.ReLU(),     
        )

        self.mu = nn.Linear(100, action_dim)
        self.log_std = nn.Linear(100, action_dim)

        self.checkpoint_path = os.path.join(checkpoint_dir, file_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.to(self.device)


    def forward(self, x):
        v = self.net(x)
        mu = self.mu(v)
        log_std = self.log_std(v)

        log_std = torch.clamp(log_std, -20, 2)

        return mu, log_std

    def sample(self, state, log_prob = True):
        mu, log_std = self.forward(state)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mu, std)


        # sampling with the reparametrization trick i.e z = mu + std * epsilon
        z = dist.rsample()

        action = torch.tanh(z)

        if not log_prob:
            return action

        log_prob = dist.log_prob(z)

        # summing over action dimension 
        log_prob = log_prob.sum(dim=-1, keepdim=True)


        # Numericaly stable tanh correction
        correction =  (2 * (
            torch.log(torch.tensor(2, device=z.device)) - z - torch.nn.functional.softplus(-2 * z)
            )
        ).sum(dim=-1, keepdim=True)    

        return action, log_prob - correction

    

    def save_checkpoint(self):
        os.makedirs(os.path.dirname(self.checkpoint_path), exist_ok=True)
        torch.save(self.state_dict(), self.checkpoint_path)

    def load_checkpoint(self):
        self.load_state_dict(torch.load(self.checkpoint_path))



class Critic(nn.Module):
    def __init__(self, state_dim, action_dim, checkpoint_dir = "./sac", filename = "critic.pth") -> None:
        super().__init__()

        self.net = nn.Sequential(
                    nn.Linear(state_dim + action_dim, 500),
                    nn.ReLU(),
                    nn.Linear(500, 100),
                    nn.ReLU(),
                    nn.Linear(100, 1)    
                )

        self.checkpoint_path = os.path.join(checkpoint_dir, filename)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.to(self.device)


        def forward(self, state, action):
            return self.net(torch.cat([state, action], dim=-1))


        def save_checkpoint(self):
            os.makedirs(os.path.dirname(self.checkpoint_path), exist_ok=True)
            torch.save(self.state_dict(), self.checkpoint_path)

        def load_checkpoint(self):
            self.load_state_dict(torch.load(self.checkpoint_path))




In [3]:
class ReplayBuffer:
    def __init__(self, size = 1000) -> None:
        self.N = size
        self.buffer = []

    def push(self, state, action, reward, new_state, done):
        if(len(self.buffer) > self.N):
            self.buffer.pop(0)
        self.buffer.append([state, action, reward, new_state, done])

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

In [ ]:
class SACAgent:
    def __init__(self, env_name, buffer_size = 1000, checkpoint_dir = "./sac") -> None:

        self.env_name = env_name # needed later for acting

        self.env = gym.make(self.env_name)

        state_dim = self.env.observation_space.shape[0]
        action_dim = self.env.action_space.shape[0]
    
        self.policy_actor = PolicyNetwork(state_dim, action_dim, checkpoint_dir)

        self.critic1 = Critic(state_dim, action_dim, checkpoint_dir, filename="critic1.pth")
        self.critic2 = Critic(state_dim, action_dim, checkpoint_dir, filename="critic2.pth")

        self.target_critic1 = Critic(state_dim, action_dim, checkpoint_dir, filename="target_critic1.pth")
        self.target_critic2 = Critic(state_dim, action_dim, checkpoint_dir, filename="target_critic2.pth")

        # initializing target critic with the critic parameters.
        self.target_critic1.load_state_dict(self.critic1.state_dict())
        self.target_critic2.load_state_dict(self.critic2.state_dict())

        self.buffer = ReplayBuffer(size = buffer_size)

        self.log_alpha = torch.zeros(1, requires_grad=True)

        self.alpha = self.log_alpha.exp()

        self.H_target = -self.env.action_space.shape[0]

        self.device = "cuda" if torch.cuda.is_available() else "cpu"




    def act(self):
        self.env = gym.make(self.env_name, render_mode="human")

        total_reward = 0.0

        done = False
        
        observation, info = self.env.reset()

        while not done:
            with torch.no_grad():
                action, _ = self.policy_actor(
                    torch.as_tensor(
                        observation,
                        dtype=torch.float32,
                        device=self.policy_actor.device
                    )
                )
                action = action.cpu().numpy()

            observation, reward, terminated, truncated, info = self.env.step(action)

            done = terminated or truncated

            total_reward += float(reward)

        self.env.close()

        return total_reward



    def train(self, batch_size = 64, iterations = 50, discount = 0.9):
        rewards = []
        for i in range(iterations):
            observation, info = self.env.reset()
            total_reward = 0.0
            done = False

            while not done:
                action = self.policy_actor.sample(
                    torch.as_tensor(observation, device=self.device ),
                    log_prob=False
                    
                )

                action = action.cpu().numpy()

                new_observation, reward, terminated, truncated, info = self.env.step(action)

                done = terminated or truncated

                self.buffer.push(observation, action, reward, new_observation, done)


                if len(self.buffer) > batch_size:
                    # sampling batch
                    batch = self.buffer.sample(batch_size)
                    st, at, rt, stnew, dt = zip(*batch)
                    st = torch.tensor(st, dtype=torch.float32, device=self.device)
                    at = torch.tensor(at, dtype=torch.float32, device=self.device)
                    rt = torch.tensor(rt, dtype=torch.float32, device=self.device)
                    stnew = torch.tensor(stnew, dtype=torch.float32, device=self.device)
                    dt = torch.tensor(dt, dtype=torch.bool, device=self.device)

                    with torch.no_grad():
                        atnew, prob_corrections = self.policy_actor.sample(stnew)

                        qmin = torch.min(
                            self.target_critic1(st,atnew),
                            self.target_critic2(st, atnew)
                        )

                        y = rt +  discount * (1 - dt.float()) * (qmin - self.alpha * prob_corrections)


In [5]:
agent = SACAgent("Humanoid-v5")

In [ ]:
total_reward = agent.act()